# Hanoi (C++)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch14_recursion/Hanoi_cpp.ipynb)

Chapter 14 — Recursion. Runnable notebook of `Hanoi`: the algorithm in C++, run on the examples of the chapter, with the «Try it» cell to run it on your own data. The pseudo-code, the examples and the proofs are on the [chapter page](https://fabiofurini.github.io/algorithms/recursion/14-recursion/).

The same notebook in [Python](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch14_recursion/Hanoi_python.ipynb) and [Java](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch14_recursion/Hanoi_java.ipynb); all the algorithms of the chapter in the [chapter notebook](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch14_recursion_cpp.ipynb).

**How to run it.** Click the «Open in Colab» badge, then *Runtime → Run all* (Ctrl+F9), or run one cell at a time with Shift+Enter, from top to bottom. Each C++ program is written to a file by a `%%writefile` cell, then compiled with `g++` and run in the next cell: everything happens inside Colab.

**Names.** There are no arrays in this chapter: variables and parameters have the same names as in the pseudo-code. The stack and the counters are global variables, and the trace is switched on with `trace = true`.

## Towers of Hanoi: Hanoi

**Input:** the number of disks $n \ge 1$, the source peg $from$, the destination peg $to$ and the auxiliary peg $aux$. **Output:** the sequence of moves that transfers disks $1, \dots, n$ from $from$ to $to$. $M(n) = 2^n - 1$ moves (optimal), $S(n) = 3 \cdot 2^n - 4 = \Theta(2^n)$ steps, depth $n$.

Pseudo-code: section 14.8 of the [chapter page](https://fabiofurini.github.io/algorithms/recursion/14-recursion/).

In [1]:
%%writefile hanoi.cpp
#include <cstdarg>
#include <cstdio>
#include <cmath>
#include <initializer_list>
#include <string>
using namespace std;

long long cnt[20];      // cnt[r] = number of executions of line r (numbering of the lecture notes)
long long calls = 0;    // number of calls = nodes of the tree of recursive calls
int maxDepth = 0;       // depth of the recursion = maximum height of the stack
long long mults = 0;    // multiplications (FastPower)
long long moves = 0;    // moves (Hanoi)
int ev = 0;             // counter of the events of the trace
int stk[1000];          // the call stack: the arguments of the active calls (at most 1000)
int sp = 0;             // number of active calls
long long seen[64];     // seen[x] = number of calls with argument x
bool trace = false;     // true: print the trace of the execution

void Reset()
{
    for (int r = 0; r < 20; r++) cnt[r] = 0;
    for (int k = 0; k < 64; k++) seen[k] = 0;
    calls = 0; maxDepth = 0; mults = 0; moves = 0; ev = 0; sp = 0;
}

long long Total()
{
    long long s = 0;
    for (int r = 0; r < 20; r++) s += cnt[r];
    return s;
}

void PrintSteps(initializer_list<int> lines)
{
    printf("Line | Executions\n");
    for (int r : lines) printf("%4d | %10lld\n", r, cnt[r]);
    printf("Total steps S = %lld\n", Total());
}

string Str(const char* fmt, ...)       // like sprintf, but returns a string
{
    char buf[256];
    va_list ap;
    va_start(ap, fmt);
    vsnprintf(buf, sizeof buf, fmt, ap);
    va_end(ap);
    return string(buf);
}

void Push(int x)                       // a call starts: its frame goes on top of the stack
{
    calls++;
    stk[sp++] = x;
    if (sp > maxDepth) maxDepth = sp;
    if (x >= 0 && x < 64) seen[x]++;
}

void Pop()                             // the call returns: its frame leaves the stack
{
    sp--;
}

string StackStr()                      // "[4, 3, 2]"
{
    string s = "[";
    for (int k = 0; k < sp; k++)
    {
        if (k > 0) s += ", ";
        s += to_string(stk[k]);
    }
    return s + "]";
}

void Event(const string& text)         // numbered line of the trace, with the stack
{
    ev++;
    printf("%2d. %s   stack: %s\n", ev, text.c_str(), StackStr().c_str());
}

void Say(const string& call, const string& action)   // line of the trace: the call, indented one level for every active call
{
    printf("%-28s| %s\n", (string(2 * (sp - 1), ' ') + call).c_str(), action.c_str());
}

bool showMoves = false;                            // true: print the moves

void Move(int disk, const char* from, const char* to)   // "move disk d from from to to": counts the move and, if showMoves, prints it
{
    moves++;
    if (showMoves) printf("%lld. move disk %d from %s to %s\n", moves, disk, from, to);
}

// Hanoi(n, from, to, aux)
void Hanoi(int n, const char* from, const char* to, const char* aux)
{
    Push(n); cnt[2]++;
    string call = trace ? Str("Hanoi(%d,%s,%s,%s)", n, from, to, aux) : "";
    if (n == 1)
    {
        cnt[3]++;
        Move(1, from, to);
        if (trace) Say(call, Str("prints: move disk 1 from %s to %s, returns", from, to));
    }
    else
    {
        if (trace) Say(call, Str("calls Hanoi(%d,%s,%s,%s)", n - 1, from, aux, to));
        cnt[5]++;
        Hanoi(n - 1, from, aux, to);
        cnt[6]++;
        Move(n, from, to);
        if (trace) Say(call, Str("prints: move disk %d from %s to %s, calls Hanoi(%d,%s,%s,%s)", n, from, to, n - 1, aux, to, from));
        cnt[7]++;
        Hanoi(n - 1, aux, to, from);
        if (trace) Say(call, "returns");
    }
    Pop();
}

#ifndef SOLO_LIBRERIA
int main()
{
    // Example of the lecture notes: Hanoi(3, A, C, B), the moves
    Reset();
    showMoves = true;
    Hanoi(3, "A", "C", "B");
    showMoves = false;
    printf("%lld moves, %lld calls, depth of the recursion = %d\n", moves, calls, maxDepth);

    // The same execution, call by call (indentation = depth in the stack)
    printf("\n");
    Reset();
    trace = true;
    printf("%-28s| %s\n", "call", "action");
    Hanoi(3, "A", "C", "B");
    trace = false;
    PrintSteps({2, 3, 5, 6, 7});
    printf("S(n) = 3 * 2^n - 4 = %d\n", 3 * (1 << 3) - 4);

    // Moves and steps: M(n) = 2^n - 1 and S(n) = 3 * 2^n - 4
    printf("\n");
    printf("  n |  moves | 2^n - 1 |    calls |   S(n) | 3 * 2^n - 4 | depth\n");
    for (int k = 1; k <= 10; k++)
    {
        Reset();
        Hanoi(k, "A", "C", "B");
        printf("%3d | %6lld | %7d | %8lld | %6lld | %11d | %d\n", k, moves, (1 << k) - 1, calls, Total(), 3 * (1 << k) - 4, maxDepth);
    }

    // The legend: 64 disks, one move per second
    printf("\n");
    printf("n = 64: 2^64 - 1 seconds are more than %.1e years\n", (pow(2.0, 64) - 1) / (365.25 * 24 * 3600));
    return 0;
}
#endif

Writing hanoi.cpp


In [2]:
!g++ -std=c++17 -O2 -o hanoi hanoi.cpp && ./hanoi

1. move disk 1 from A to C
2. move disk 2 from A to B
3. move disk 1 from C to B
4. move disk 3 from A to C
5. move disk 1 from B to A
6. move disk 2 from B to C
7. move disk 1 from A to C
7 moves, 7 calls, depth of the recursion = 3

call                        | action
Hanoi(3,A,C,B)              | calls Hanoi(2,A,B,C)
  Hanoi(2,A,B,C)            | calls Hanoi(1,A,C,B)
    Hanoi(1,A,C,B)          | prints: move disk 1 from A to C, returns
  Hanoi(2,A,B,C)            | prints: move disk 2 from A to B, calls Hanoi(1,C,B,A)
    Hanoi(1,C,B,A)          | prints: move disk 1 from C to B, returns
  Hanoi(2,A,B,C)            | returns
Hanoi(3,A,C,B)              | prints: move disk 3 from A to C, calls Hanoi(2,B,C,A)
  Hanoi(2,B,C,A)            | calls Hanoi(1,B,A,C)
    Hanoi(1,B,A,C)          | prints: move disk 1 from B to A, returns
  Hanoi(2,B,C,A)            | prints: move disk 2 from B to C, calls Hanoi(1,A,C,B)
    Hanoi(1,A,C,B)          | prints: move disk 1 from A to C, returns
 

**Try it yourself.** Change the number of disks and count the moves.

In [3]:
%%writefile prova_hanoi.cpp
#define SOLO_LIBRERIA   // do not run the main of hanoi.cpp
#include "hanoi.cpp"

int main()
{
    int n = 4;        // <-- change the number of disks
    Reset();
    showMoves = true;
    Hanoi(n, "A", "C", "B");
    showMoves = false;
    printf("%lld moves = 2^%d - 1 = %d\n", moves, n, (1 << n) - 1);
    return 0;
}

Writing prova_hanoi.cpp


In [4]:
!g++ -std=c++17 -O2 -o prova_hanoi prova_hanoi.cpp && ./prova_hanoi

1. move disk 1 from A to B
2. move disk 2 from A to C
3. move disk 1 from B to C
4. move disk 3 from A to B
5. move disk 1 from C to A
6. move disk 2 from C to B
7. move disk 1 from A to B
8. move disk 4 from A to C
9. move disk 1 from B to C
10. move disk 2 from B to A
11. move disk 1 from C to A
12. move disk 3 from B to C
13. move disk 1 from A to B
14. move disk 2 from A to C
15. move disk 1 from B to C
15 moves = 2^4 - 1 = 15


---

Teaching material by [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza University of Rome.
Text, figures and data [CC BY 4.0](https://github.com/fabiofurini/algorithms/blob/main/LICENSE), code [MIT](https://github.com/fabiofurini/algorithms/blob/main/LICENSE-CODE).